<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 06 — Capstone: SQL Agent (ReAct vs Agentic Loop)

A **minimal, runnable test**: point a local model at a real SQL database through an MCP
server and check that both multi-step agent types actually drive it correctly.

This notebook builds:

1. A tiny synthetic SQLite database (`customers`, `products`, `orders`).
2. A minimal **read-only SQL MCP server** exposing `list_tables` / `describe_table` / `run_sql`
   — the same pattern used in [05 — Capstone: Insight Engine](05_capstone_insight_engine.ipynb),
   but self-contained here (no external support files).
3. The **same question**, asked twice, against the **same tools**:
   - **`ReactAgent`** — the text-parsing Thought/Action/Observation loop ([01](01_providers_and_agents.ipynb) §5)
   - **`AgenticLoop`** — the native tool-calling loop ([01](01_providers_and_agents.ipynb) §4)
4. A **ground-truth check**: the expected numbers computed directly with `sqlite3`, so you can
   eyeball whether each agent's answer is actually correct — not just "it ran".

**Why MCP instead of a hand-rolled SQL tool?** Neurosurfer already speaks MCP natively
(see [04 — MCP Servers](04_mcp_servers.ipynb)) — a `McpTool` slots into a `ToolPool` exactly
like a built-in tool, with zero framework code to write. There's no need to build a custom
`run_sql` `Tool` subclass (or reach for LangChain's SQL toolkit) when three `@server.tool()`
functions do the job.

**Model used:** any **native-tool-calling** local model works for both variants (`AgenticLoop`
needs it; `ReactAgent` doesn't but happily uses the same model — see [01](01_providers_and_agents.ipynb)
§5). Validated here with `qwen/qwen3.5-9b` via LM Studio on `localhost:1234` (the same model
validated for the SQL analyst node in [05](05_capstone_insight_engine.ipynb)).

---

## 1. Setup

Point Python at the repo root and confirm the `mcp` extra is installed:

```bash
pip install "neurosurfer[mcp]"
```

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, sqlite3, random, tempfile
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

try:
    import mcp
    print("mcp SDK: OK")
except ImportError:
    print("mcp SDK missing — run: pip install 'neurosurfer[mcp]'")

# Gitignored scratch dir (tutorials/tmp/ already exists and is git-ignored).
TMP_DIR = repo_root / "tutorials" / "tmp"
TMP_DIR.mkdir(exist_ok=True)

version,1.0.0 | python 3.11.13
os,Linux 6.17.0-35-generic (x86_64)
torch,2.9.1+cu129 CUDA: yes (12.9)
mps,no (built: False)
transformers,4.57.6 sentEmb 5.1.0
accelerate,1.12.0 bnb 0.47.0
gpu,NVIDIA GeForce RTX 3080 Ti


neurosurfer 1.0.0
mcp SDK: OK


In [2]:
# ── LM Studio connection ──────────────────────────────────────────────────────
# Make sure LM Studio is running with a tool-calling model loaded and the local
# server ON (port 1234). AgenticLoop requires native tool calling; ReactAgent
# works with any model but we use the same one for a fair side-by-side.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"   # adjust to your loaded model — needs tool_use capability
CONTEXT_WINDOW  = 48_000

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")
print(f"Native tool calling: {provider.capabilities.tool_call_style}")

Provider: qwen/qwen3.5-9b
Native tool calling: openai


---

## 2. JupyterIO

Agents (and the MCP gate) need an `IOHandler` for approvals. Our tools are all
**read-only** (`readOnlyHint=True`), so with the default `mcp_policy="gated"` none of
these prompts actually fire — this is just the standard notebook boilerplate from
[01](01_providers_and_agents.ipynb) / [04](04_mcp_servers.ipynb).

In [3]:
class JupyterIO:
    """Auto-approving IOHandler for notebook demos."""

    async def ask(self, question: str, options=None) -> str:
        return "yes"

    async def request_plan_approval(self, plan: str):
        return True, ""

    async def request_shell_approval(self, command: str, reason: str) -> bool:
        return True

    async def request_write_approval(self, path: str, summary: str) -> str:
        return "once"

    def notify(self, message: str) -> None:
        pass

print("JupyterIO ready.")

JupyterIO ready.


---

## 3. A minimal SQL test database

Three small tables — enough to need a **join** and a **group by**, not so much that the
point of the notebook (testing the agent loop) gets lost. Seeded with `random.seed(42)`
so it's reproducible. Pure `sqlite3` + stdlib `random` — no pandas/numpy needed.

In [4]:
DB_PATH = TMP_DIR / "shop.db"
random.seed(42)

cities = ["Lahore", "Karachi", "Islamabad", "London", "New York"]
categories = {
    "Electronics": ["Laptop", "Phone", "Headphones", "Monitor"],
    "Books":       ["Novel", "Cookbook", "Biography"],
    "Home":        ["Lamp", "Chair", "Rug"],
}

customers = [(i, f"Customer {i}", random.choice(cities)) for i in range(1, 21)]

products, pid = [], 1
for cat, names in categories.items():
    for name in names:
        products.append((pid, name, cat, round(random.uniform(15, 500), 2)))
        pid += 1

orders, oid = [], 1
for _ in range(200):
    cust = random.choice(customers)[0]
    prod = random.choice(products)
    qty = random.randint(1, 5)
    order_date = f"2025-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}"
    orders.append((oid, cust, prod[0], qty, order_date))
    oid += 1

con = sqlite3.connect(DB_PATH)
con.execute("DROP TABLE IF EXISTS orders")
con.execute("DROP TABLE IF EXISTS products")
con.execute("DROP TABLE IF EXISTS customers")
con.execute("CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
con.execute("CREATE TABLE products (product_id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL)")
con.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, "
            "product_id INTEGER, quantity INTEGER, order_date TEXT)")
con.executemany("INSERT INTO customers VALUES (?,?,?)", customers)
con.executemany("INSERT INTO products VALUES (?,?,?,?)", products)
con.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders)
con.commit()
con.close()

print(f"shop.db -> {DB_PATH}")
print(f"  customers: {len(customers)}   products: {len(products)}   orders: {len(orders)}")

shop.db -> /home/nomi/workspace/neurosurfer/tutorials/tmp/shop.db
  customers: 20   products: 10   orders: 200


### Ground truth

Compute the real answer directly with `sqlite3` — no agent involved — so we have something
to check the agents' answers against later.

In [5]:
con = sqlite3.connect(DB_PATH)

print("Revenue by category:")
for row in con.execute("""
    SELECT p.category, ROUND(SUM(p.price * o.quantity), 2) AS revenue
    FROM orders o JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category ORDER BY revenue DESC
"""):
    print(" ", row)

print("\nCustomers by city:")
for row in con.execute(
    "SELECT city, COUNT(*) FROM customers GROUP BY city ORDER BY COUNT(*) DESC"
):
    print(" ", row)

print(f"\nTotal customers: {con.execute('SELECT COUNT(*) FROM customers').fetchone()[0]}")
con.close()

Revenue by category:
  ('Home', 63510.46)
  ('Electronics', 49260.94)
  ('Books', 43666.0)

Customers by city:
  ('Lahore', 8)
  ('New York', 5)
  ('Karachi', 5)
  ('London', 1)
  ('Islamabad', 1)

Total customers: 20


---

## 4. A minimal read-only SQL MCP server

Three tools, exactly the pattern from [05](05_capstone_insight_engine.ipynb):
`list_tables`, `describe_table`, `run_sql` — the last one rejects anything that isn't a
`SELECT`/`PRAGMA`, so the database is safe to hand to a local model. Written to a temp file
and launched as a **stdio** subprocess (same trick as [04](04_mcp_servers.ipynb) §3).

In [6]:
SQL_MCP_SERVER_SRC = r'''
import sqlite3
import sys

from mcp.server.fastmcp import FastMCP

server = FastMCP("sql-agent-tutorial")
DB_PATH = sys.argv[1] if len(sys.argv) > 1 else "shop.db"


@server.tool(annotations={"readOnlyHint": True}, description="List every table in the database.")
def list_tables() -> str:
    con = sqlite3.connect(DB_PATH)
    try:
        rows = con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        return ", ".join(r[0] for r in rows) or "(no tables)"
    finally:
        con.close()


@server.tool(annotations={"readOnlyHint": True}, description="Return the column names and types of a table.")
def describe_table(table: str) -> str:
    con = sqlite3.connect(DB_PATH)
    try:
        rows = con.execute(f"PRAGMA table_info({table})").fetchall()
        if not rows:
            return f"No such table: {table}"
        return "\n".join(f"{r[1]} {r[2]}" for r in rows)
    finally:
        con.close()


@server.tool(annotations={"readOnlyHint": True}, description="Run a read-only SELECT/PRAGMA query; returns up to 50 rows as text.")
def run_sql(query: str) -> str:
    q = query.strip().lower()
    if not (q.startswith("select") or q.startswith("pragma")):
        return "Only SELECT/PRAGMA queries are allowed."
    con = sqlite3.connect(DB_PATH)
    try:
        cur = con.execute(query)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchmany(50)
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        return f"{header}\n{body}" if body else f"{header}\n(no rows)"
    except Exception as e:
        return f"SQL error: {e}"
    finally:
        con.close()


if __name__ == "__main__":
    server.run("stdio")
'''

SERVER_PATH = str(TMP_DIR / "sql_mcp_server.py")
Path(SERVER_PATH).write_text(SQL_MCP_SERVER_SRC, encoding="utf-8")
print("wrote", SERVER_PATH)

wrote /home/nomi/workspace/neurosurfer/tutorials/tmp/sql_mcp_server.py


---

## 5. Connect via `McpSession`

We want the connection to stay alive across **several cells** (both agent variants reuse it),
so `McpSession` — the synchronous wrapper from [05](05_capstone_insight_engine.ipynb) §5 — is
the right tool here, rather than `McpManager`'s `async with` block from
[04](04_mcp_servers.ipynb) (which tears the session down at the end of one cell/task).

In [7]:
from neurosurfer.config.mcp import McpServerConfig
from neurosurfer.mcp import McpSession

session = McpSession([McpServerConfig(
    name="shop",
    transport="stdio",
    command=sys.executable,
    args=[SERVER_PATH, str(DB_PATH)],
)])
for s in session.connect():
    print(f"server {s.name!r}: connected={s.connected}, tools={s.tools}")

server 'shop': connected=True, tools=['list_tables', 'describe_table', 'run_sql']


In [8]:
from neurosurfer.tools import ToolPool, ToolContext
from neurosurfer.tools.builtin import FinishTool

pool = ToolPool([*session.tools(), FinishTool()])
ctx = ToolContext(cwd=repo_root, io=JupyterIO())
print("pool:", pool.names())

pool: ['list_tables', 'describe_table', 'run_sql', 'finish']


---

## 6. The test question

The **same** question, **same** system prompt, **same** tool pool, for both variants below —
so any difference in the final answer comes from the agent loop, not from the setup. The
system prompt spells out one `run_sql` query per sub-question so a small local model doesn't
take a shortcut and guess a number instead of looking it up.

In [9]:
question = (
    "Using the SQL tools, tell me: (1) how many customers are there in total, "
    "(2) which product category generates the most total revenue (price * quantity), "
    "and (3) which city has the most customers. Give concrete numbers for each."
)

SQL_SYSTEM_PROMPT = (
    "You are a careful SQL analyst with access to list_tables, describe_table, and run_sql. "
    "Never guess a number — every figure in your answer must come from a run_sql result. "
    "Run three separate run_sql queries before answering: "
    "(a) SELECT COUNT(*) FROM customers; "
    "(b) a query joining orders and products, grouped by category, to find total revenue "
    "(price * quantity) per category; "
    "(c) a query grouping customers by city to find the city with the most customers. "
    "Only give your final answer once you have all three results. If a query returns a SQL "
    "error, immediately fix it (e.g. check column names with describe_table) and retry — "
    "never give up on a sub-question after one failed query."
)
print(question)

Using the SQL tools, tell me: (1) how many customers are there in total, (2) which product category generates the most total revenue (price * quantity), and (3) which city has the most customers. Give concrete numbers for each.


---

## 7. Variant A — `ReactAgent` (text-parsing ReAct)

Prompts the model to emit `Thought` / `Action` / `Action Input`, parses that text, executes
the tool, feeds back an `Observation`. No native tool-calling API required — this is the
loop to reach for on models that don't support (or poorly support) JSON function calling.
`verbose=False` so we render the trace ourselves, exactly as in [01](01_providers_and_agents.ipynb) §5.

In [10]:
from neurosurfer.agents import ReactAgent, Guardrails, events
from neurosurfer.llm.types import GenerationConfig

react_agent = ReactAgent(
    provider      = provider,
    tools         = pool,
    system_prompt = SQL_SYSTEM_PROMPT,
    guardrails    = Guardrails(max_turns=16),
    io            = JupyterIO(),
    cwd           = repo_root,
    gen_config    = GenerationConfig(max_tokens=2048, temperature=0.2),
    verbose       = False,
)

print(f"Task: {question}\n{'─' * 60}")

react_answer = ""
thinking = False

async for ev in react_agent.run(question):
    if isinstance(ev, events.ToolStarted):
        if thinking:
            print(); thinking = False
        print(f"🔧 {ev.title or ev.name}({ev.args})")
    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:150].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 150 else ""
        print(f"   ↳ {preview}{suffix}")
    elif isinstance(ev, events.ThinkingDelta):
        if not thinking:
            print("💭 Thinking…", end="", flush=True)
            thinking = True
    elif isinstance(ev, events.TextDelta):
        if thinking:
            print("\n"); thinking = False
        react_answer += ev.text
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")
        react_answer = react_answer.strip() or ev.report or ""

Task: Using the SQL tools, tell me: (1) how many customers are there in total, (2) which product category generates the most total revenue (price * quantity), and (3) which city has the most customers. Give concrete numbers for each.
────────────────────────────────────────────────────────────


💭 Thinking…


🔧 shop: list_tables…({})
   ↳ customers, products, orders
💭 Thinking…


🔧 shop: run_sql…({'query': 'SELECT COUNT(*) FROM customers'})
   ↳ COUNT(*) 20


💭 Thinking…

...

with

8

 customers

)



────────────────────────────────────────────────────────────
Run finished — completed


---

## 8. Variant B — `AgenticLoop` (native tool calling)

Uses the provider's native function-calling API — no text parsing. Ends when the model calls
`finish` or reaches `max_turns`. This is the production default for models that support it
([01](01_providers_and_agents.ipynb) §4).

In [11]:
from neurosurfer.agents import AgenticLoop

agentic_loop = AgenticLoop(
    provider      = provider,
    tools         = pool,
    system_prompt = SQL_SYSTEM_PROMPT + " Call finish with your findings once you're done.",
    guardrails    = Guardrails(max_turns=16),
    io            = JupyterIO(),
    cwd           = repo_root,
    gen_config    = GenerationConfig(max_tokens=2048, temperature=0.2),
    verbose       = False,
)

print(f"Task: {question}\n{'─' * 60}")

loop_answer = ""

async for ev in agentic_loop.run(question):
    if isinstance(ev, events.ToolStarted):
        print(f"🔧 {ev.name}({ev.args})")
    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:150].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 150 else ""
        print(f"   ↳ {preview}{suffix}")
    elif isinstance(ev, events.TextDelta):
        loop_answer += ev.text
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")
        if ev.report:
            print(f"Report: {ev.report}")
        loop_answer = loop_answer.strip() or ev.report or ""

Task: Using the SQL tools, tell me: (1) how many customers are there in total, (2) which product category generates the most total revenue (price * quantity), and (3) which city has the most customers. Give concrete numbers for each.
────────────────────────────────────────────────────────────


🔧 list_tables({})
   ↳ customers, products, orders




────────────────────────────────────────────────────────────
Run finished — completed


---

## 9. Side-by-side vs. ground truth

Eyeball both agents' numbers against the direct-SQL ground truth from §3. If a model is
prone to arithmetic slips, this is where it shows up — the *tools* returned correct rows either
way, so any mismatch here is a reasoning error, not a plumbing error.

In [12]:
print("GROUND TRUTH (direct SQL, §3):")
con = sqlite3.connect(DB_PATH)
for row in con.execute("""
    SELECT p.category, ROUND(SUM(p.price * o.quantity), 2) AS revenue
    FROM orders o JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category ORDER BY revenue DESC LIMIT 1
"""):
    print(f"  top category by revenue: {row[0]} (${row[1]:,.2f})")
for row in con.execute(
    "SELECT city, COUNT(*) FROM customers GROUP BY city ORDER BY COUNT(*) DESC LIMIT 1"
):
    print(f"  top city by customer count: {row[0]} ({row[1]} customers)")
print(f"  total customers: {con.execute('SELECT COUNT(*) FROM customers').fetchone()[0]}")
con.close()

print("\n" + "=" * 60)
print("REACTAGENT answer:\n")
print(react_answer.strip() or "(empty)")

print("\n" + "=" * 60)
print("AGENTICLOOP answer:\n")
print(loop_answer.strip() or "(empty)")

GROUND TRUTH (direct SQL, §3):
  top category by revenue: Home ($63,510.46)
  top city by customer count: Lahore (8 customers)
  total customers: 20

REACTAGENT answer:

Here are the concrete numbers you requested:

1. **Total customers**: 20

2. **Product category with most total revenue**: Home (with total revenue of $63,510.46)

3. **City with the most customers**: Lahore (with 8 customers)

AGENTICLOOP answer:

(empty)


---

## 10. Cleanup

Disconnect from the MCP server (stops the subprocess and the background session thread).

In [13]:
session.close()
print("MCP session closed.")

MCP session closed.


---

## Summary

| Step | API |
|---|---|
| Describe the SQL server | `McpServerConfig(name, transport="stdio", command, args=[script, db_path])` |
| Keep it alive across cells | `McpSession(servers)` → `.connect()` / `.tools()` / `.close()` |
| Build the pool | `ToolPool([*session.tools(), FinishTool()])` |
| Text-parsing ReAct | `ReactAgent(provider, tools=pool, ...)` → `async for ev in agent.run(q)` |
| Native tool calling | `AgenticLoop(provider, tools=pool, ...)` → `async for ev in agent.run(q)` |

**Key ideas**

- MCP turns three plain Python functions into tools **any** agent type can call — `ReactAgent`
  and `AgenticLoop` see the exact same `ToolPool`, so this notebook isolates the *agent loop*
  as the only variable between the two runs.
- `ReactAgent` doesn't need native tool-calling support, but it's driven through the same
  permission-gated tool execution path as `AgenticLoop` — so a tool written once (or, as here,
  served once over MCP) works with either loop unmodified.
- A ground-truth `sqlite3` query is cheap insurance: it tells you whether a wrong answer is a
  **tool-calling failure** (never got the right rows) or a **reasoning failure** (had the right
  rows, did the arithmetic wrong).

### What's next

- **[01 — Providers & Agents](01_providers_and_agents.ipynb):** `ReactAgent` vs `AgenticLoop` in depth.
- **[04 — MCP Servers](04_mcp_servers.ipynb):** connecting to real MCP servers, permissions, persistence.
- **[05 — Capstone: Insight Engine](05_capstone_insight_engine.ipynb):** the same SQL-over-MCP
  pattern as one node in a larger graph.
- **Make it yours:** point `McpServerConfig` at Postgres/MySQL via a community MCP server instead
  of this SQLite demo — the agent code doesn't change.